# 07 - Reconhecimento de Entidades Nomeadas (NER) com BERTugues

Neste notebook, vamos treinar (Fine-Tuning) o modelo `ricardoz/BERTugues-base-portuguese-cased` para a tarefa de **Reconhecimento de Entidades Nomeadas (NER)**.

Utilizaremos a renomada base **LeNER-Br** (Legal Named Entity Recognition in Portuguese), que consiste em textos jurídicos brasileiros anotados com entidades como PESSOA, ORGANIZACAO, LEGISLACAO, JURISPRUDENCIA, etc.

## 1. Instalando e Importando as Bibliotecas Necessárias

In [1]:
# !pip install transformers datasets evaluate accelerate seqeval

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate

## 2. Carregando e Preparando o Dataset LeNER-Br

Vamos usar uma função customizada para fazer o download e parsear os arquivos `.conll` originais do LeNER-Br, evitando problemas com a biblioteca datasets atual.

In [2]:
model_name = "ricardoz/BERTugues-base-portuguese-cased"

import urllib.request
from datasets import Dataset, DatasetDict

def load_lener_br_split(url):
    response = urllib.request.urlopen(url)
    lines = response.read().decode('utf-8').splitlines()
    texts, labels = [], []
    current_tokens, current_tags = [], []
    for line in lines:
        line = line.strip()
        if not line:
            if current_tokens:
                texts.append(current_tokens)
                labels.append(current_tags)
                current_tokens, current_tags = [], []
        else:
            parts = line.split(" ")
            if len(parts) == 2:
                current_tokens.append(parts[0])
                current_tags.append(parts[1])
    if current_tokens:
        texts.append(current_tokens)
        labels.append(current_tags)
    return {"tokens": texts, "ner_tags": labels}

# Baixando e processando os dados do LeNER-Br
urls = {
    "train": "https://raw.githubusercontent.com/peluz/lener-br/master/leNER-Br/train/train.conll",
    "validation": "https://raw.githubusercontent.com/peluz/lener-br/master/leNER-Br/dev/dev.conll",
    "test": "https://raw.githubusercontent.com/peluz/lener-br/master/leNER-Br/test/test.conll"
}

print("Baixando dados do LeNER-Br...")
train_data = load_lener_br_split(urls["train"])
validation_data = load_lener_br_split(urls["validation"])

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(validation_data)
})

# Obtendo a lista única de rótulos
unique_tags = set(tag for doc in train_data["ner_tags"] for tag in doc)
label_list = sorted(list(unique_tags))
if "O" in label_list:
    label_list.remove("O")
    label_list = ["O"] + label_list

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in id2label.items()}

train_texts = [list(x) for x in dataset["train"]["tokens"]]
train_labels = dataset["train"]["ner_tags"]
train_labels_ids = [[label2id[tag] for tag in doc] for doc in train_labels]

val_texts = [list(x) for x in dataset["validation"]["tokens"]]
val_labels = dataset["validation"]["ner_tags"]
val_labels_ids = [[label2id[tag] for tag in doc] for doc in val_labels]

print(f"Exemplos no treino: {len(train_texts)}, Exemplos na validação: {len(val_texts)}")

Baixando dados do LeNER-Br...
Exemplos no treino: 7827, Exemplos na validação: 1176


## 3. Tokenização e Alinhamento de Subtokens

O BERT utiliza tokens de subpalavras (WordPiece). Isso significa que uma única palavra (e, por consequência, um único rótulo) pode ser dividida em múltiplos tokens pelo modelo.

Para treinar o modelo, precisamos de uma função que:
1. Tokenize as listas de palavras (`is_split_into_words=True`).
2. Atribua o rótulo da palavra ao primeiro subtoken.
3. Ignore os demais subtokens da mesma palavra e os tokens especiais `[CLS]` e `[SEP]`, usando o valor `-100` (que o PyTorch ignora automaticamente no cálculo da função de perda/loss).

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(texts, labels_list):
    tokenized_inputs = tokenizer(
        texts, 
        truncation=True, 
        is_split_into_words=True, 
        padding="max_length", 
        max_length=128,
        return_tensors="pt"
    )

    aligned_labels = []
    for i, label in enumerate(labels_list):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Mapeia token -> palavra original
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Tokens especiais [CLS] e [SEP] ou padding
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Primeiro subtoken de uma nova palavra
                label_ids.append(label[word_idx])
            else:
                # Subtokens subsequentes da mesma palavra
                label_ids.append(-100)
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = torch.tensor(aligned_labels)
    return tokenized_inputs

train_texts = [[str(t) for t in doc] for doc in train_texts]
val_texts = [[str(t) for t in doc] for doc in val_texts]
tokenized_dataset = tokenize_and_align_labels(train_texts, train_labels_ids)

print("Tamanho dos input_ids:", tokenized_dataset["input_ids"].shape)
print("Exemplo de alinhamento das labels (o ID -100 é ignorado):\n", tokenized_dataset["labels"][0].tolist())

Tamanho dos input_ids: torch.Size([7827, 128])
Exemplo de alinhamento das labels (o ID -100 é ignorado):
 [-100, 0, -100, -100, 0, 0, -100, -100, -100, -100, 0, -100, -100, -100, 0, 0, -100, -100, 0, 0, -100, -100, -100, -100, -100, 0, -100, 0, -100, -100, 0, -100, -100, 0, 0, -100, -100, -100, -100, 0, 0, -100, -100, -100, -100, 0, -100, -100, 4, -100, -100, -100, -100, -100, -100, 10, -100, -100, -100, -100, -100, 0, 0, -100, -100, 0, -100, -100, -100, -100, 0, 0, -100, -100, -100, -100, 0, 0, -100, -100, -100, -100, -100, 0, 0, -100, -100, -100, -100, -100, -100, 0, 0, -100, -100, -100, -100, 0, 0, -100, -100, -100, -100, -100, -100, 0, 0, -100, -100, 0, 0, -100, -100, -100, -100, 0, 0, -100, -100, -100, 0, 0, -100, -100, -100, -100, -100, -100]


Vamos encapsular nosso dataset como um objeto `torch.utils.data.Dataset` simples para que o `Trainer` o aceite sem reclamar.

In [4]:
class NERDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: val[idx].clone().detach() for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["labels"])

# Dataset de treino
train_dataset = NERDataset(tokenized_dataset)

# Dataset de validação
tokenized_val = tokenize_and_align_labels(val_texts, val_labels_ids)
val_dataset = NERDataset(tokenized_val)

## 4. Carregando o Modelo e Loop de Treinamento

In [5]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Removemos os -100 (subtokens ignorados) para calcular a métrica corretamente
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

training_args = TrainingArguments(
    output_dir="./bertugues-ner-lenerbr",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    num_train_epochs=4,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("\nIniciando o Treinamento no LeNER-Br...")
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 


Iniciando o Treinamento no LeNER-Br...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.076641,0.793403,0.837508,0.814859,0.977337
2,0.124062,0.096780,0.772207,0.865608,0.816244,0.976909
3,0.024531,0.088854,0.829128,0.883323,0.855368,0.980230
4,0.013413,0.096231,0.824032,0.883934,0.852933,0.979748


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1960, training_loss=0.04342368792514412, metrics={'train_runtime': 177.6193, 'train_samples_per_second': 176.265, 'train_steps_per_second': 11.035, 'total_flos': 2045373107770368.0, 'train_loss': 0.04342368792514412, 'epoch': 4.0})

## 5. Inferência com o Modelo Treinado

Agora que as camadas convolucionais / head foram brevemente adaptadas aos nossos dados de exemplo (toy), podemos testar com uma frase.

In [6]:
texto_teste = "O Supremo Tribunal Federal (STF) decidiu a favor da União na ação movida pela Petrobras em Brasília."

inputs = tokenizer(texto_teste, return_tensors="pt")
device = model.device
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits

predictions = torch.argmax(logits, dim=2)
predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(f"Frase de Teste: {texto_teste}\n")
print("Predições por token:")
for token, prediction in zip(tokens, predicted_token_class):
    if token not in ['[CLS]', '[SEP]', '[PAD]']:
        print(f"{token:<12} -> {prediction}")

Frase de Teste: O Supremo Tribunal Federal (STF) decidiu a favor da União na ação movida pela Petrobras em Brasília.

Predições por token:
O            -> O
Supremo      -> B-ORGANIZACAO
Tribunal     -> I-ORGANIZACAO
Federal      -> I-ORGANIZACAO
(            -> O
STF          -> B-ORGANIZACAO
)            -> O
decidiu      -> O
a            -> O
favor        -> O
da           -> O
União        -> B-ORGANIZACAO
na           -> O
ação         -> O
movida       -> O
pela         -> O
Petrobras    -> B-ORGANIZACAO
em           -> O
Brasília     -> B-LOCAL
.            -> O
